## Match ratings workshop
This notebook reproduces the CM preprocessing pipeline and then applies the precomputed PCA weights and reference means/stds to generate per-match ratings.
### Loading the data
Load the raw match JSON and set the project root for file access.

In [91]:
# Loading the data
import json
import pandas as pd
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from sklearn.covariance import MinCovDet
import numpy as np

# Define the project root (one folder up from /workshop)
project_root = Path("..").resolve()

# Add the project root to sys.path so Jupyter can find your 'src' module
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Load data/valencia_cf_1/matches.json
matches_path = project_root / "tests" / "fixtures" / "testing_data" / "data" / "valencia_cf_1" / "matches.json"
raw_matches = json.load(open(matches_path))
print(f"Loaded {len(raw_matches)} matches from {matches_path}")

Loaded 99 matches from C:\Users\kazik\Projects\Gaffers-Clipboard\tests\fixtures\testing_data\data\valencia_cf_1\matches.json


### Flattening and filtering to CM
We normalize player_performances, map team context (team_possession, team_xg, opponent_xg, team_passes) for Valencia CF, drop GK rows, and keep only CM performances.

In [92]:
normalized_df = pd.json_normalize(
    raw_matches,
    record_path="player_performances",
    meta=[
        "id", ["data", "half_length"], 
        ["data", "home_team_name"], 
        ["data", "away_team_name"], 
        ["data", "home_stats", "possession"], 
        ["data", "away_stats", "possession"],
        ["data", "home_stats", "xg"],
        ["data", "away_stats", "xg"],
        ["data", "home_stats", "passes"],
        ["data", "away_stats", "passes"]
    ]
)
# Create 4 new columns: team_possession, team_xg, opponent_xg, and team_passes, and set all the values to NaN
normalized_df["team_possession"] = float("nan")
normalized_df["team_xg"] = float("nan")
normalized_df["opponent_xg"] = float("nan")
normalized_df["team_passes"] = float("nan")

for index, performance in normalized_df.iterrows():
    # Check if the performance is for the home team or away team
    if performance["data.home_team_name"] == "Valencia CF":
        normalized_df.at[index, "team_possession"] = performance["data.home_stats.possession"]
        normalized_df.at[index, "team_xg"] = performance["data.home_stats.xg"]
        normalized_df.at[index, "opponent_xg"] = performance["data.away_stats.xg"]
        normalized_df.at[index, "team_passes"] = performance["data.home_stats.passes"]
    else:
        normalized_df.at[index, "team_possession"] = performance["data.away_stats.possession"]
        normalized_df.at[index, "team_xg"] = performance["data.away_stats.xg"]
        normalized_df.at[index, "opponent_xg"] = performance["data.home_stats.xg"]
        normalized_df.at[index, "team_passes"] = performance["data.away_stats.passes"]

# Remove the home and away names, possession, xg, and passes columns
normalized_df = normalized_df.drop(columns=["data.home_team_name", "data.away_team_name", "data.home_stats.possession", "data.away_stats.possession", "data.home_stats.xg", "data.away_stats.xg", "data.home_stats.passes", "data.away_stats.passes"])

normalized_df = normalized_df.rename(columns={"id": "match_id"})

normalized_df = normalized_df[normalized_df["performance_type"] != "GK"]

normalized_df = normalized_df.dropna(axis=1, how="all")

# Now we want to filter for players with only "CM" in "positions_played"
cm_players_df = normalized_df[normalized_df["positions_played"].apply(lambda x: x == ["CM"])]

# Now remove columns that are all NaN
cm_players_df = cm_players_df.dropna(axis=1, how="all")

# Remove "performance_type" and "positions_played" columns
cm_players_df = cm_players_df.drop(columns=["performance_type", "positions_played"])

# Move "match_id" to the front, and rename the id column to "performance_id"
cols = cm_players_df.columns.tolist()
cols.insert(0, cols.pop(cols.index("match_id")))
cm_players_df = cm_players_df[cols]
cm_players_df.head()

# Rename "data.half_length" to "half_length"
cm_players_df = cm_players_df.rename(columns={"data.half_length": "half_length"})

# Output number of rows and columns, and the first few rows of the resulting dataframe
print(f"Number of rows: {cm_players_df.shape[0]}")
print(f"Number of columns: {cm_players_df.shape[1]}")
print("Columns:")
print(cm_players_df.columns.tolist())
print("\n")

# Output the max and min of every column, to get a sense of the range of values
for col in cm_players_df.columns:
    if col in ["match_id", "player_id"]:
        continue
    if cm_players_df[col].dtype in ["int64", "float64"]:
        print(f"{col}: min={cm_players_df[col].min()}, max={cm_players_df[col].max()}")

Number of rows: 269
Number of columns: 24
Columns:
['match_id', 'player_id', 'goals', 'assists', 'shots', 'shot_accuracy', 'passes', 'pass_accuracy', 'dribbles', 'dribble_success_rate', 'tackles', 'tackle_success_rate', 'offsides', 'fouls_committed', 'possession_won', 'possession_lost', 'minutes_played', 'distance_covered', 'distance_sprinted', 'half_length', 'team_possession', 'team_xg', 'opponent_xg', 'team_passes']


goals: min=0.0, max=8.0
assists: min=0.0, max=8.0
shots: min=0.0, max=12.0
shot_accuracy: min=0.0, max=100.0
passes: min=0.0, max=61.0
pass_accuracy: min=0.0, max=100.0
dribbles: min=0.0, max=54.0
dribble_success_rate: min=0.0, max=100.0
tackles: min=0.0, max=12.0
tackle_success_rate: min=0.0, max=100.0
offsides: min=0.0, max=8.0
fouls_committed: min=0.0, max=8.0
possession_won: min=0.0, max=8.0
possession_lost: min=0.0, max=14.0
minutes_played: min=2.0, max=94.0
distance_covered: min=0.3, max=14.1
distance_sprinted: min=0.0, max=8.0
team_possession: min=39.0, max=71.0


### Fixing data types
Convert all non-ID columns to numeric so downstream math (normalization, smoothing, z-scores) is safe.

In [93]:
# 1. Select every column EXCEPT your identifiers
cols_to_convert = cm_players_df.columns.drop(['match_id', 'player_id'])

# 2. Force them all to numeric in one swoop
cm_players_df[cols_to_convert] = cm_players_df[cols_to_convert].apply(pd.to_numeric, errors='coerce')

# Optional: If any weird JSON text like "N/A" got turned into a NaN, 
# you can safely fill them with 0s to keep the math from breaking later.
cm_players_df[cols_to_convert] = cm_players_df[cols_to_convert].fillna(0)

### Data quality filters
We remove very short appearances (minutes_played < 10) and apply volume masks for percentage stats so tiny samples do not create misleading rates.
Thresholds: passes >= 5, shots >= 2, dribbles >= 3, tackles >= 2.

In [94]:
# Step 1: Removing players with less than 10 minutes played
cm_players_df = cm_players_df[cm_players_df["minutes_played"] >= 10]

In [95]:
# Step 1: Volume masking
# For each percentage stat, their associated volume attempted stat must reach a certain threshold for the percentage 
# stat to be considered valid. For example, if a player attempted 0 passes, their pass accuracy should not be considered valid,
# and should be set to the mean.
volume_perc_pairs = [
    ("passes", "pass_accuracy"),
    ("shots", "shot_accuracy"),
    ("dribbles", "dribble_success_rate"),
    ("tackles", "tackle_success_rate")
]
volume_masks = {
    "passes": 5,
    "shots": 2,
    "dribbles": 3,
    "tackles": 2
}

for volume_col, perc_col in volume_perc_pairs:
    # apply the mask
    valid = cm_players_df[cm_players_df[volume_col] >= volume_masks[volume_col]]
    mean_value = valid[perc_col].mean()
    cm_players_df[perc_col] = cm_players_df.apply(
        lambda r: r[perc_col] if r[volume_col] >= volume_masks[volume_col] else mean_value,
        axis=1
    )

### Normalizing by half length and minutes (Bayesian smoothing)
We need per-90 stats that are comparable across half lengths and uneven minutes. The pipeline is:
1) Half-length standardization to $H_{base}=10$:
$$X_{adj} = X_{raw} \times \left( \frac{H_{base}}{H_{match}} \right)$$
2) Bayesian smoothing with $d=30$ dummy minutes to reduce small-sample volatility. We blend the observed rate with a prior rate so short appearances do not produce extreme per-90s:
$$r_{smoothed} = \frac{M_{played}}{M_{played} + d} r_{obs} + \frac{d}{M_{played}+d} r_{prior}$$
- Rare stats (goals, assists, shots) use $r_{prior}=0$.
- Volume stats use $r_{prior}$ equal to the league average per-90 computed from the adjusted dataset.<br>

**Example (rare stat):** 1 shot in 5 minutes gives $r_{obs}=18$ per 90. With $d=30$, $r_{smoothed} \approx 2.57$ shots per 90.<br>
**Example (volume stat):** League passes $=40$ per 90, player makes 5 passes in 5 minutes ($r_{obs}=90$). With $d=30$, $r_{smoothed} \approx 47.1$ passes per 90.<br>
1) Convert to per-90 using the smoothed rate:
$$X_{p90} = \frac{X_{adj} + r_{prior} \times \left( \frac{d}{90} \right)}{M_{played} + d} \times 90$$
This keeps small samples from overpowering the rating while letting full-match performances dominate.

In [96]:
# ---------------------------------------------------------
# Step 1: Standardize Absolute Volume (The Half-Length Fix)
# ---------------------------------------------------------
cum_columns = ["goals", "assists", "shots", "passes", "dribbles", "tackles", 
               "possession_won", "possession_lost", "fouls_committed", "offsides", 
               "distance_covered", "distance_sprinted"]

h_base = 10.0

for col in cum_columns:
    # We overwrite the raw stat to represent a standard 10-minute half game.
    # This ensures 5-min half games don't corrupt our dataset averages later.
    cm_players_df[col] = cm_players_df[col] * (h_base / cm_players_df["half_length"])


# ---------------------------------------------------------
# Step 2: Set Bayesian Smoothing Parameters
# ---------------------------------------------------------
dummy_minutes = 30.0  # Shrinkage anchor (requires ~a half of football to break away from mean)


# ---------------------------------------------------------
# Step 3: Apply Smoothed Per-90 Scaling
# ---------------------------------------------------------
# A. Rare Stats (Assume 0 for dummy minutes)
rare_cols = ["goals", "assists", "shots"]
for col in rare_cols:
    cm_players_df[f"{col}_p90"] = (cm_players_df[col] / (cm_players_df["minutes_played"] + dummy_minutes)) * 90.0


# B. Volume Stats (Assume dataset average for dummy minutes)
volume_cols = ["passes", "dribbles", "tackles", "possession_won", "possession_lost", 
               "fouls_committed", "offsides", "distance_covered", "distance_sprinted"]

for col in volume_cols:
    # 1. Calculate the true, half-length-adjusted dataset average Per 90
    league_avg_p90 = (cm_players_df[col].sum() / cm_players_df["minutes_played"].sum()) * 90.0
    
    # 2. Calculate the dummy stat allocation for 45 minutes
    dummy_stat = league_avg_p90 * (dummy_minutes / 90.0)
    
    # 3. Apply the final smoothed formula
    cm_players_df[f"{col}_p90"] = ((cm_players_df[col] + dummy_stat) / (cm_players_df["minutes_played"] + dummy_minutes)) * 90.0

# Print to verify
print(f"Passes mean after smoothing: {cm_players_df['passes_p90'].mean():.2f}, std: {cm_players_df['passes_p90'].std():.2f}")

Passes mean after smoothing: 35.49, std: 6.74


### Expected Threat proxy
We add an xT-style bonus using per-90 workrate ratio and passing volume/accuracy:
$$Bonus_{xT} = 0.25 \times \left( \frac{D_{sprinted,p90}}{D_{covered,p90}} \right) \times \ln(Pass_{acc} \times Passes_{p90} + 1)$$
This rewards high-intensity running paired with secure, high-volume distribution.

In [97]:
cm_players_df["xT_bonus_p90"] = 0.25 * (cm_players_df["distance_sprinted_p90"] / cm_players_df["distance_covered_p90"]) * np.log(cm_players_df["pass_accuracy"] * cm_players_df["passes_p90"] + 1)

**Adjusting for match supremacy**<br>
Prior analysis suggests a meaningful share of rating variance is team-dependent, so we add a match context adjustment based on xG:
$$\Delta_{xG} = \frac{xG_{team} + 1}{xG_{opponent} + 1}$$
If $\Delta_{xG} < 1$ (team was out-created), a given raw score is more valuable than the same output in a dominant team. We apply a small rating offset:
$$R_{final} = R_{raw} - \gamma \ln(\Delta_{xG})$$
Here $\gamma = 0.2$, keeping the adjustment conservative.
The scalar is computed now and subtracted after the sigmoid mapping so ratings reflect match context.

In [98]:
# Match supremacy scalar
gamma = 0.2
delta_xg = (cm_players_df['team_xg'] + 1) / (cm_players_df['opponent_xg'] + 1)
match_supremacy_scalar = gamma * np.log(delta_xg)

### Load PCA weights
These weights come from the PCA calibration notebook (including block scaling). This notebook uses them as fixed coefficients, so the preprocessing steps here must match the calibration pipeline.

In [ ]:
# Now we import weights from root/workshop/position_weights.json
# Which is in the format {"CB": {"goals_p90": 0.3, ...}, "CM": {...}, "ST": {...}}
col_names = ["goals_p90", "assists_p90", "shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xT_bonus_p90"]
position_weights_path = project_root / "workshop" / "position_weights.json"
with open(position_weights_path, "r") as f:
    position_weights = json.load(f)
weights_dict = position_weights["CM"]
final_weights = np.array([weights_dict[col] for col in col_names])

### Load reference means and standard deviations
These are the feature means/stds from the same preprocessing pipeline. They define the z-score baseline used for scoring.

In [ ]:
# Now we import the means and stds from root/workshop/position_means_stds.json
# Which is in the format {"CB": {"goals_p90": {"mean": 0.1, "std": 0.2}, ...}, "CM": {...}, "ST": {...}}
position_means_stds_path = project_root / "workshop" / "position_means_stds.json"
with open(position_means_stds_path, "r") as f:
    position_means_stds = json.load(f)
means_stds_dict = position_means_stds["CM"]

### Creating the final match rating algorithm — z-score computation
Here we convert each per-90 feature into a standardized z-score using the precomputed reference means and standard deviations. The reference values come from the calibration pipeline (workshop/position_means_stds.json, CM section).
For a feature $x$, mean $u$ and standard deviation $\sigma$, the z-score is computed as:
$$z = \frac{x - u}{\sigma}$$
When a feature represents a negative contribution (e.g. fouls committed, possession lost, offsides) we invert the direction so higher z is always better:
$$z_{neg} = \frac{u - x}{\sigma}$$
Implementation notes:
- If $\sigma=0$ we set $z=0$ to avoid division-by-zero artifacts.
- The code applies the sign inversion only to the small set `negative_stats = ['fouls_committed_p90', 'possession_lost_p90', 'offsides_p90']` so all z-scores are aligned in the 
 sense.
- These z-scores are the canonical standardized inputs for the weighting step that follows; no weighting or aggregation is performed in this cell.

In [101]:
# Compute final z-scores
negative_stats = ["fouls_committed_p90", "possession_lost_p90", "offsides_p90"]
for col in col_names:
    mean = means_stds_dict[col]["mean"]
    std = means_stds_dict[col]["std"]
    if std == 0:
        cm_players_df[f"{col}_z"] = 0
    elif col in negative_stats:
        cm_players_df[f"{col}_z"] = (mean - cm_players_df[col]) / std
    else:
        cm_players_df[f"{col}_z"] = (cm_players_df[col] - mean) / std

### Floors and caps on z-scores (detailed)
We apply targeted clipping to z-scores to stabilize the rating scale and limit pathological influence from very small samples or extreme outliers.
The rules and rationale:
- Zero-penalty floor for attacking stats: negative attacking z-scores are set to 0 so a non-attacking CM is not actively penalised by the attacking dimension. This keeps the role-neutral baseline sensible and prevents defensive-minded players from being driven to low totals by missing attacking volume.
- Efficiency floor for accuracies: pass and tackle accuracy z-scores are clipped at a lower bound (≥ -0.75) so a single failed action or short sample cannot create an extreme negative penalty. This reduces volatility when minutes are low and accuracy estimates are noisy.
- Caps on goals/assists z-scores: we cap these at 1.5 because we add explicit manual bonuses for goals/assists later. The cap prevents a single large raw z-score from overwhelming the final 0–10 mapping even before bonuses are applied.<br><br>

Practical effect examples:
- If a player's `goals_p90_z` would be 2.4 from a short-but-productive sample, it is capped to 1.5 and the remaining contribution is instead expressed via the defined goal bonus (controlled, proportional).
- If a player's `pass_accuracy_z` from a short cameo is -1.3, the efficiency floor raises it to -0.75, reducing the undue penalty from noise.
These choices are pragmatic: they trade some theoretical fidelity for robustness and a stable, interpretable rating distribution.

In [102]:
# 1. Universal "Zero-Penalty" Floor for CM Attacking Stats
# If a defensive CM doesn't attack, they simply get 0 points, not negative points.
attacking_stats = ['goals_p90_z', 'assists_p90_z', 'shots_p90_z', 'shot_accuracy_z']
for col in attacking_stats:
    cm_players_df[col] = np.clip(cm_players_df[col], a_min=0.0, a_max=None)

# 2. Universal Efficiency Floor
# Prevents missing 1 pass or 1 tackle from causing a mathematical black hole
efficiency_stats = ['tackle_success_rate_z', 'pass_accuracy_z']
for col in efficiency_stats:
    cm_players_df[col] = np.clip(cm_players_df[col], a_min=-0.75, a_max=None)

# 3. Cap Goal/Assist Z-Scores
# Because you are adding massive manual bonuses (0.8 and 0.6) later, 
# we cap the raw Z-score here so a hat-trick doesn't break the 10.0 scale.
cm_players_df["goals_p90_z"] = np.clip(cm_players_df["goals_p90_z"], a_min=None, a_max=1.5)
cm_players_df["assists_p90_z"] = np.clip(cm_players_df["assists_p90_z"], a_min=None, a_max=1.5)

### Weighted raw score — aggregation formula
After z-scores are computed and clipped, the baseline aggregate score for each player is the dot product between their standardized feature vector $Z_i$ (1 x p) and the positional weight vector $\lambda_{pos}$ (p x 1):
$$S_i = Z_i \cdot \lambda_{pos} = \sum_{j=1}^{p} Z_{ij} \lambda_{pos,j}$$
In matrix form (applied to all players at once):
$$S = Z \cdot \lambda^T$$
The code implements this as a vectorized matrix multiplication using NumPy's `dot` (i.e. `np.dot(z_matrix, final_weights)`), where `z_matrix` has a row per player and `final_weights` is the column vector of PCA-derived coefficients. This cell computes only the aggregation — bonus adjustments happen in subsequent cells.

In [103]:
z_score_cols = [f"{col}_z" for col in col_names]
z_matrix = cm_players_df[z_score_cols].values
cm_players_df['raw_score'] = np.dot(z_matrix, final_weights)

### Targeted bonuses for standout events (expanded)
The PCA-based aggregate captures broad, consistent contributions. We add small, interpretable bonuses to reward unmistakable match events that the PCA may only partially capture or that we want to emphasize explicitly.
Rules applied in the code:
- Goals: if `goals_p90_z` > 0, add `0.8 * goals_p90_z` to `raw_score`. This scales the bonus with how unusual the goal rate is relative to the reference distribution.
- Assists: if `assists_p90_z` > 0, add `0.6 * assists_p90_z` similarly.<br><br>
- High-volume play bonuses: for `passes_p90_z`, `dribbles_p90_z`, and `possession_won_p90_z`, add `0.2 * (z - 2)` when the z-score exceeds 2.0. This only rewards truly exceptional volume (>2 SD above mean).
Design rationale and examples:
  - Scaling by the z-score lets the bonus respect both frequency and extremity: a small positive z yields a small bonus; a large z yields a proportionally larger bonus, but still bounded by earlier caps and the sigmoid mapping.
  - The choice of 0.8/0.6 for goals/assists is deliberately larger than the 0.2 pass/dribble bonus because direct goal contributions are more decisive for match outcomes. Caps on goals/assists z-scores (1.5) bound the possible effect before bonus is applied.
  - Example: a player with `goals_p90_z = 1.0` receives +0.8 to `raw_score`. If the same player had `passes_p90_z = 2.5`, they receive an additional `(2.5 - 2) * 0.2 = 0.1` from passes. These bonuses layering produces a controlled uplift on top of the PCA baseline.

In [ ]:
# Add bonuses for goals and assists only when the z-score is positive
cm_players_df['raw_score'] += np.where(cm_players_df['goals_p90_z'] > 0, cm_players_df['goals_p90_z'] * 0.8, 0)
cm_players_df['raw_score'] += np.where(cm_players_df['assists_p90_z'] > 0, cm_players_df['assists_p90_z'] * 0.6, 0)

# Same for passes, dribbles, and possession won, where z-score is over 2, using smaller bonuses (0.4 or 0.2)
cm_players_df['raw_score'] += np.where(cm_players_df['passes_p90_z'] > 2, (cm_players_df['passes_p90_z'] - 2) * 0.4, 0)
cm_players_df['raw_score'] += np.where(cm_players_df['dribbles_p90_z'] > 2, (cm_players_df['dribbles_p90_z'] - 2) * 0.2, 0)
cm_players_df['raw_score'] += np.where(cm_players_df['possession_won_p90_z'] > 2, (cm_players_df['possession_won_p90_z'] - 2) * 0.4, 0)

To map the unbounded continuous variable $S_i \in (-\infty, \infty)$ to the recognized $0.0 - 10.0$ football rating continuum, we apply the inverse logistic function discussed previously.<br>
As identified in the commercial analysis, WhoScored utilizes a base of 6.0 while Sofascore utilizes 6.5. For this custom algorithm, an exact median baseline of $6.0$ is targeted for an entirely average performance (where the sum of standard deviations $S_i = 0$).
$$R_{raw} = 10 \times \left( \frac{1}{1 + e^{-k(S_i - S_0)}} \right)$$
We solve for the translation parameter $S_0$ such that $R_{raw} = 6.0$ when $S_i = 0$:
$$6.0 = \frac{10}{1 + e^{k \cdot S_0}}$$
$$1 + e^{k \cdot S_0} = \frac{10}{6.0} = 1.666$$
$$e^{k \cdot S_0} = 0.666$$
$$k \cdot S_0 = \ln(0.666) \approx -0.405$$
Setting the steepness parameter $k = 0.85$ provides an optimal curve. This specific slope allows for $3\sigma$ events (such as a multi-goal performance) to mathematically approach scores of 9.5 without hitting a rigid asymptote of 10.0 too early in the distribution. Solving for $S_0$ with $k=0.85$ yields $S_0 \approx -0.477$.<br><br>
Thus, the final formula for the raw rating before the match supremacy adjustment is:
$$R_{raw} = 10 \times \left( \frac{1}{1 + e^{-0.85(S_i + 0.477)}} \right)$$

In [105]:
k = 0.85
s_0 = np.log(2/3) / k
# Create a new column, applying the sigmoid transformation to the raw score
cm_players_df['raw_rating'] = 10 * (1 / (1 + np.exp(-k * (cm_players_df['raw_score'] - s_0))))

### Apply match supremacy adjustment and clamp
We subtract the match supremacy scalar, then clamp the final rating to the 0-10 range.

In [106]:
# Now create a final rating column that subtracts the match supremacy scalar from the raw rating
cm_players_df['final_rating'] = cm_players_df['raw_rating'] - match_supremacy_scalar

In [107]:
# Now a final clamp, to ensure that the final rating is between 0 and 10
cm_players_df['final_rating'] = cm_players_df['final_rating'].apply(lambda x: max(0, min(10, x)))

In [108]:
print(cm_players_df.iloc[-1])

match_id                          99
player_id                         24
goals                            1.0
assists                          0.0
shots                            3.0
shot_accuracy                  100.0
passes                          61.0
pass_accuracy                   95.0
dribbles                        54.0
dribble_success_rate            94.0
tackles                          7.0
tackle_success_rate             29.0
offsides                         0.0
fouls_committed                  1.0
possession_won                   2.0
possession_lost                  4.0
minutes_played                  91.0
distance_covered                13.5
distance_sprinted                6.6
half_length                       10
team_possession                 61.0
team_xg                          5.8
opponent_xg                      1.0
team_passes                    271.0
goals_p90                   0.743802
assists_p90                      0.0
shots_p90                   2.231405
p

### Inspect a sample row
Print a random row to sanity-check the pipeline and final rating values.

In [109]:
# Randomly pick a row, and output every column, in the format "column_name: value,", with one per line
random_row = cm_players_df.sample(n=1).iloc[0]
for col in cm_players_df.columns:
    print(f"{col}: {random_row[col]:.2f},")

match_id: 76.00,
player_id: 24.00,
goals: 0.00,
assists: 0.00,
shots: 2.00,
shot_accuracy: 50.00,
passes: 20.00,
pass_accuracy: 85.00,
dribbles: 19.00,
dribble_success_rate: 95.00,
tackles: 5.00,
tackle_success_rate: 60.00,
offsides: 0.00,
fouls_committed: 0.00,
possession_won: 3.00,
possession_lost: 5.00,
minutes_played: 91.00,
distance_covered: 13.30,
distance_sprinted: 6.40,
half_length: 10.00,
team_possession: 40.00,
team_xg: 3.30,
opponent_xg: 1.60,
team_passes: 165.00,
goals_p90: 0.00,
assists_p90: 0.00,
shots_p90: 1.49,
passes_p90: 23.71,
dribbles_p90: 21.47,
tackles_p90: 4.96,
possession_won_p90: 2.88,
possession_lost_p90: 5.06,
fouls_committed_p90: 0.05,
offsides_p90: 0.01,
distance_covered_p90: 12.85,
distance_sprinted_p90: 5.87,
xT_bonus_p90: 0.87,
goals_p90_z: 0.00,
assists_p90_z: 0.00,
shots_p90_z: 0.25,
shot_accuracy_z: 0.00,
passes_p90_z: -1.71,
pass_accuracy_z: -0.46,
dribbles_p90_z: -0.87,
dribble_success_rate_z: 0.48,
tackles_p90_z: -0.16,
tackle_success_rate_z: 1.27,